In [0]:
import mlflow
import mlflow.pyfunc
import pandas as pd
import requests
import json
import time
import re



class ClassificadorComentarios(mlflow.pyfunc.PythonModel):
    """
    Classifica comentários de turismo em: elogio, critica, duvida, outros.
    Usa a API do OpenRouter com o modelo Llama 3.1 8B (gratuito).
    Processa em lotes pra economizar chamadas.
    """

    CATEGORIAS_VALIDAS = {"elogio", "critica", "duvida", "outros"}
    TAMANHO_LOTE = 10  # quantos comentários por chamada

    def load_context(self, context):
        with open(context.artifacts["config"], "r") as f:
            config = json.load(f)
        self.api_key = config["api_key"]
        self.modelo = config["modelo"]
        self.url = "https://openrouter.ai/api/v1/chat/completions"

    def _montar_prompt(self, descricoes):
        comentarios_numerados = "\n".join(
            f"{i+1}. {d}" for i, d in enumerate(descricoes)
        )
        return f"""Você é um classificador de comentários turísticos. Classifique cada comentário abaixo em UMA das categorias:

- elogio: expressa satisfação, aprovação ou positividade
- critica: expressa insatisfação, reclamação ou negatividade
- duvida: é uma pergunta ou pedido de informação
- outros: não se encaixa nas três anteriores

Responda APENAS no formato "número. categoria", uma classificação por linha, sem explicações.

Comentários:
{comentarios_numerados}

Respostas:"""

    def _chamar_api(self, prompt, tentativa=0):
        try:
            resp = requests.post(
                self.url,
                headers={
                    "Authorization": f"Bearer {self.api_key}",
                    "Content-Type": "application/json",
                },
                json={
                    "model": self.modelo,
                    "messages": [{"role": "user", "content": prompt}],
                    "max_tokens": 200,
                    "temperature": 0,
                },
                timeout=60,
            )

            # Rate limit: espera e tenta de novo (até 3 vezes)
            if resp.status_code == 429 and tentativa < 3:
                espera = 2 ** (tentativa + 3)  # 8s, 16s, 32s
                print(f"Rate limit. Aguardando {espera}s...")
                time.sleep(espera)
                return self._chamar_api(prompt, tentativa + 1)

            resp.raise_for_status()
            return resp.json()["choices"][0]["message"]["content"]
        except Exception as e:
            print(f"Erro na API: {e}")
            return ""

    def _parsear_resposta(self, texto_resposta, qtd_esperada):
        """Extrai 'número. categoria' de cada linha da resposta."""
        resultado = ["outros"] * qtd_esperada
        # Regex tolerante: captura número e depois qualquer palavra
        padrao = re.compile(r"(\d+)[.\):\-\s]+(\w+)")
        
        for match in padrao.finditer(texto_resposta.lower()):
            idx = int(match.group(1)) - 1
            categoria = match.group(2)
            # Normaliza acentos
            categoria = (
                categoria.replace("í", "i")
                         .replace("ú", "u")
                         .replace("ó", "o")
                         .replace("á", "a")
                         .replace("ê", "e")
            )
            if 0 <= idx < qtd_esperada and categoria in self.CATEGORIAS_VALIDAS:
                resultado[idx] = categoria
        return resultado

    def _classificar_lote(self, descricoes):
        # Limpa descrições inválidas
        descricoes_limpas = [
            str(d) if d and isinstance(d, str) and d.strip() else ""
            for d in descricoes
        ]
        
        # Se todas forem vazias, já retorna
        if not any(descricoes_limpas):
            return ["outros"] * len(descricoes)
        
        prompt = self._montar_prompt(descricoes_limpas)
        resposta = self._chamar_api(prompt)
        categorias = self._parsear_resposta(resposta, len(descricoes_limpas))
        
        # Pequena pausa entre lotes pra não estourar rate limit
        time.sleep(1)
        return categorias

    def predict(self, context, model_input: pd.DataFrame) -> pd.Series:
        descricoes = model_input["descricao"].tolist()
        resultados = []
        
        for i in range(0, len(descricoes), self.TAMANHO_LOTE):
            lote = descricoes[i:i + self.TAMANHO_LOTE]
            resultados.extend(self._classificar_lote(lote))
        
        return pd.Series(resultados)